In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../generated_data/feature_extraction/all_results.csv")

In [3]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Demo_interest_in_politics_Symbol,Demo_interest_in_politics_Speculation,Demo_marital_status_Category,Demo_marital_status_Explanation,Demo_marital_status_Symbol,Demo_marital_status_Speculation,Demo_partisan_loyalties_Category,Demo_partisan_loyalties_Explanation,Demo_partisan_loyalties_Symbol,Demo_partisan_loyalties_Speculation
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,i4),20.0,"single never married, including those who are ...",the users tweets contain no references to a sp...,m2),60.0,independent closer to the democrats,while the user frequently attacks republican p...,pid3),40.0
23986,142827500,2024-09-08 00:00:00+00:00,"['8 weeks to election', {'text': ""@diveomusic ...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,i3),40.0,"single never married, including those who are ...",the users tweets and profile contain no refere...,m2),80.0,independent closer to the democrats,the users limited political activity (election...,pid3),50.0
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,i3),40.0,"single never married, including those who are ...","no mention of a partner, wedding, or relations...",m2),80.0,independent not closer to either party,the users tweets reference elections and townh...,pid4),65.0
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,i3),45.0,"single never married, including those who are ...","there is no mention of a spouse, wedding, or r...",m2),70.0,independent not closer to either party,the user never mentions a political party or a...,pid4),70.0


In [ ]:


def count_tweets_by_pattern(x):
    """
    Count tweets by looking for the pattern of tweet dictionaries
    """
    if not isinstance(x, str) or x.strip() in ('[]', ''):
        return 0
    
    # Count occurrences of the tweet pattern: {'text': ... 'created_at': ...}
    tweet_count = x.count("'text':")
    return tweet_count

# Apply pattern counting
df['num_tweets'] = df['tweets_struct'].apply(count_tweets_by_pattern)


num_rows = len(df)
num_users = df['id'].nunique()
windows_per_user = df.groupby('id').size()
avg_windows = windows_per_user.mean()
min_windows = windows_per_user.min()
max_windows = windows_per_user.max()

empty_windows_ratio = (df['num_tweets'] == 0).mean()

print("### Dataset Overview ###")
print(f"Total rows: {num_rows:,}")
print(f"Unique users: {num_users:,}")
print(f"Avg windows per user: {avg_windows:.2f} (min={min_windows}, max={max_windows})")
print(f"Fraction of empty tweet windows: {empty_windows_ratio:.2%}\n")

# Tweet activity
tweets_per_user = df.groupby('id')['num_tweets'].sum()
avg_tweets_per_user = tweets_per_user.mean()
nonempty_windows_per_user = df.groupby('id').apply(lambda g: (g['num_tweets'] > 0).sum()).mean()

print("### Tweet Activity ###")
print(f"Average total tweets per user: {avg_tweets_per_user:.2f}")
print(f"Average non-empty windows per user: {nonempty_windows_per_user:.2f}\n")

# Speculation scores ---
spec_cols = [c for c in df.columns if c.endswith("_Speculation")]
avg_speculation_scores = df[spec_cols].mean().sort_values(ascending=False)

print("### Average Speculation Scores ###")
print(avg_speculation_scores)
print("")

# Category distributions
cat_cols = [c for c in df.columns if c.endswith("_Category")]
print("### Category Distributions ###")
for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(5))
    missing_pct = df[col].isna().mean() * 100
    print(f"Missing: {missing_pct:.1f}%")

# summary table
summary = {
    "num_rows": num_rows,
    "num_users": num_users,
    "avg_windows_per_user": avg_windows,
    "empty_windows_ratio": empty_windows_ratio,
    "avg_tweets_per_user": avg_tweets_per_user,
}
summary_df = pd.DataFrame([summary])
print("\n### Summary Table ###")
print(summary_df)


### Dataset Overview ###
Total rows: 23,990
Unique users: 928
Avg windows per user: 25.85 (min=2, max=26)
Fraction of empty tweet windows: 56.27%

### Tweet Activity ###
Average total tweets per user: 1595.03
Average non-empty windows per user: 11.31

### Average Speculation Scores ###
Demo_household_income_bracket_Speculation                                                                                                                                                   63.782215
Demo_marital_status_Speculation                                                                                                                                                             63.707968
Belief_opinion_on_whether_divorce_in_the_usa_should_be_easier_or_more_difficult_to_obtain_than_it_is_Speculation                                                                            59.995806
Belief_opinion_on_whether_courts_in_the_users_s_area_deal_too_harshly_or_not_harshly_enough_with_criminals_Speculation 

C:\Users\yuyue\AppData\Local\Temp\ipykernel_32548\1049538652.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nonempty_windows_per_user = df.groupby('id').apply(lambda g: (g['num_tweets'] > 0).sum()).mean()


In [8]:
avg_speculation_scores

Demo_household_income_bracket_Speculation                                                                                                                                                   63.782215
Demo_marital_status_Speculation                                                                                                                                                             63.707968
Belief_opinion_on_whether_divorce_in_the_usa_should_be_easier_or_more_difficult_to_obtain_than_it_is_Speculation                                                                            59.995806
Belief_opinion_on_whether_courts_in_the_users_s_area_deal_too_harshly_or_not_harshly_enough_with_criminals_Speculation                                                                      55.979699
Belief_opinion_on_declining_share_of_white_people_in_america_Speculation                                                                                                                    53.806615
Belief_opi

In [6]:
print(df.info())   


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23990 entries, 0 to 23989
Columns: 228 entries, id to num_tweets
dtypes: bool(1), float64(52), int64(4), object(171)
memory usage: 41.6+ MB
None
